<a href="https://colab.research.google.com/github/dcthyun0308/ESAA/blob/main/0501_%EC%84%B8%EC%85%98_%ED%95%A9%EC%84%B1%EA%B3%B1_%EC%97%B0%EC%8A%B5%EB%AC%B8%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. CIFAR-10 데이터셋을 이용한 CNN 모델

## Step 1. 필요한 라이브러리 임포트

In [1]:
# 필요한 라이브러리 불러오기
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.autograd import Variable

In [2]:
# 디바이스 설정 (GPU 사용 가능하면 GPU, 없으면 CPU)
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available(): # Mac(M1/M2/M3) 사용자라면 MPS 사용
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"현재 사용 중인 디바이스: {device}")

현재 사용 중인 디바이스: cpu


## Step 2. CIFAR-10 데이터셋 다운로드

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # 정규화 추가
])

# 데이터셋 로드
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

100%|██████████| 170M/170M [00:11<00:00, 14.7MB/s]


##Step 3. DataLoader 설정

In [4]:
# DataLoader를 사용하여 배치 크기 64로 데이터를 로드하세요.
batch_size = 64

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=batch_size,
                          shuffle=True)

test_loader = DataLoader(dataset=test_dataset,
                         batch_size=batch_size,
                         shuffle=False)

# 설정 확인을 위한 출력
print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 782
Test batches: 157


## Step 4. CNN 모델 정의

In [5]:
class CIFAR10_CNN(nn.Module):
    def __init__(self):
        super(CIFAR10_CNN, self).__init__()

        # 첫 번째 합성곱 층
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32) # 출력 채널 수와 동일하게 설정

        # 두 번째 합성곱 층
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
        self.bn2 = nn.BatchNorm2d(64)

        # 세 번째 합성곱 층
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)
        self.bn3 = nn.BatchNorm2d(128)

        # 풀링 층 (MaxPooling)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully Connected Layer
        # 입력 이미지 32x32 -> (conv1+pool) 16x16 -> (conv2+pool) 7x7 -> (conv3+pool) 2x2
        # 최종 특징 맵 크기: 128채널 * 2 * 2 = 512
        self.fc1 = nn.Linear(128 * 2 * 2, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 10) # CIFAR-10 클래스 수: 10개
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        # 활성화 함수 ReLU 적용
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = x.view(x.size(0), -1) # Flatten

        x = F.relu(self.fc1(x)) # Fully Connected 활성화 함수 추가
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

## Step 5. 모델 학습 설정

In [6]:
# 모델을 초기화하세요.
model = CIFAR10_CNN().to(device)

# 손실 함수로 CrossEntropyLoss를 사용하고 옵티마이저는 Adam을 사용하세요.
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 설정 확인
print(model)

CIFAR10_CNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=512, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=10, bias=True)
  (dropout): Dropout(p=0.25, inplace=False)
)


## Step 6. 모델 학습 루프

In [7]:
# 모델 학습을 위한 루프를 작성하세요. 에폭 수는 2로 설정합니다.
num_epochs = 2
for epoch in range(num_epochs):
    for images, labels in train_loader:
        # 데이터를 설정된 디바이스(GPU/CPU)로 전송
        images, labels = images.to(device), labels.to(device)

        # 모델을 통해 예측값 계산 (Forward Pass)
        outputs = model(images)

        # 손실 계산
        loss = criterion(outputs, labels)

        # 경사 초기화 (Gradient를 0으로 설정)
        optimizer.zero_grad()

        # 역전파 (Backward Pass)
        loss.backward()

        # 최적화 (가중치 업데이트)
        optimizer.step()

    # 에폭마다 손실값 출력
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [1/2], Loss: 0.7174
Epoch [2/2], Loss: 0.6674


## Step 7. 테스트 정확도 평가

In [8]:
# 모델의 성능 평가
correct = 0
total = 0

# 평가 모드에서는 기울기를 계산할 필요가 없으므로 메모리 절약을 위해 torch.no_grad() 사용
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)

        # 가장 높은 확률(값)을 가진 인덱스를 예측값으로 선택
        _, predicted = torch.max(outputs.data, 1)

        # 전체 이미지 개수 누적
        total += labels.size(0)

        # 예측값과 실제 정답이 일치하는 개수 누적
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 66.97%


# 2. MNIST 데이터셋 분류 모델




## Step 1. 필요한 라이브러리 임포트

In [9]:
# 필요한 라이브러리 불러오기
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.autograd import Variable

In [10]:
# 디바이스 설정 (GPU 사용 가능하면 GPU, 없으면 CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Mac 사용자(M1/M2/M3 등 Apple Silicon)라면 아래 코드를 대신 사용할 수 있습니다.
# device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

print(f"현재 사용 중인 디바이스: {device}")

현재 사용 중인 디바이스: cpu


## Step 2. MNIST 데이터셋 다운로드

In [11]:
# MNIST 데이터셋을 다운로드하고, 텐서로 변환할 수 있도록 필요한 전처리 함수를 추가하세요.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# 데이터셋 로드
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.78MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 154kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.46MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.53MB/s]


##Step 3. DataLoader 설정

In [12]:
# DataLoader를 사용하여 배치 크기 64로 데이터를 로드하세요.
batch_size = 64

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=batch_size,
                          shuffle=True)

test_loader = DataLoader(dataset=test_dataset,
                         batch_size=batch_size,
                         shuffle=False)

## Step 4. CNN 모델 정의

In [13]:
# CNN 모델 정의
class MNIST_CNN(nn.Module):
    def __init__(self):
        super(MNIST_CNN, self).__init__()
        # MNIST는 흑백 이미지이므로 in_channels=1입니다.
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # 28x28 -> (pool) 14x14 -> (pool) 7x7
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10) # 출력은 0~9까지 10개 클래스
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = x.view(-1, 64 * 7 * 7) # Flatten
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

## Step 5. 모델 학습 설정

In [14]:
# 모델을 초기화하세요.
model = MNIST_CNN().to(device)

# 손실 함수로 CrossEntropyLoss를 사용하고 옵티마이저는 Adam을 사용하세요.
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Step 6. 모델 학습 루프

In [15]:
# 모델 학습을 위한 루프를 작성하세요. 에폭 수는 2로 설정합니다.
num_epochs = 2

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # 데이터를 설정된 디바이스(GPU/CPU)로 전송
        images = images.to(device)
        labels = labels.to(device)

        # Forward Pass: 모델에 데이터를 넣어 예측값 계산
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward Pass & Optimization: 기울기 계산 및 가중치 업데이트
        optimizer.zero_grad() # 기울기 초기화
        loss.backward()      # 역전파
        optimizer.step()     # 가중치 갱신

        if (i+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

Epoch [1/2], Step [100/938], Loss: 0.2087
Epoch [1/2], Step [200/938], Loss: 0.1296
Epoch [1/2], Step [300/938], Loss: 0.1006
Epoch [1/2], Step [400/938], Loss: 0.0277
Epoch [1/2], Step [500/938], Loss: 0.1962
Epoch [1/2], Step [600/938], Loss: 0.0728
Epoch [1/2], Step [700/938], Loss: 0.0260
Epoch [1/2], Step [800/938], Loss: 0.0324
Epoch [1/2], Step [900/938], Loss: 0.0290
Epoch [2/2], Step [100/938], Loss: 0.0271
Epoch [2/2], Step [200/938], Loss: 0.0094
Epoch [2/2], Step [300/938], Loss: 0.0101
Epoch [2/2], Step [400/938], Loss: 0.0601
Epoch [2/2], Step [500/938], Loss: 0.0080
Epoch [2/2], Step [600/938], Loss: 0.0131
Epoch [2/2], Step [700/938], Loss: 0.1150
Epoch [2/2], Step [800/938], Loss: 0.0319
Epoch [2/2], Step [900/938], Loss: 0.0638


## Step 7. 테스트 정확도 평가

In [16]:
# 모델의 성능 평가
model.eval() # 모델을 평가 모드로 전환 (드롭아웃 등 비활성화)

with torch.no_grad(): # 평가 시에는 기울기를 계산할 필요 없음
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1) # 가장 높은 확률을 가진 클래스 선택

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Test Accuracy: {100 * correct / total:.2f}%')

Test Accuracy: 98.93%
